# Data Quality Toolkit — Quickstart

A hands-on walkthrough of the core DQT Python API across four workflows:

| Step | Function | What it does |
|---|---|---|
| 1 | `profile_csv` | Structural statistics — row count, dtypes, null rates, cardinality |
| 2 | `assess_csv` | Quality issue detection + 0–100 quality score |
| 3 | `plan_csv` | Per-column preprocessing recommendations |
| 4 | `compute_univariate_chart_data` | Column-level distribution data for inline display |

> **Run this notebook from the `examples/` directory**, or update `CSV_PATH` below.

## Setup

Install the toolkit (core only):

```bash
pip install data-quality-toolkit
```

In [ ]:
from pathlib import Path

from data_quality_toolkit.api import assess_csv, plan_csv, profile_csv
from data_quality_toolkit.domain.profiling.charts import compute_univariate_chart_data

print('DQT imports OK')

## Demo dataset

`demo/sample_orders.csv` is a synthetic business orders dataset with 14 columns,
mixed data types, nullable fields, and representative quality variation.
It ships with the repository under `examples/demo/`.

In [ ]:
CSV_PATH = Path('demo/sample_orders.csv')
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f'CSV not found: {CSV_PATH}. Run from the examples/ directory.'
    )
print(f'Dataset: {CSV_PATH.resolve()}')

## Step 1 — Profile

`profile_csv` loads the CSV and returns structural statistics for the dataset and
every column. No data is written to disk.

In [ ]:
prof    = profile_csv(CSV_PATH)
profile = prof['profile']

run_id = prof['run_id']
rows   = profile['rows']
cols   = profile['cols']
mem_mb = profile['memory_mb']

print(f'Run ID  : {run_id}')
print(f'Rows    : {rows:,}')
print(f'Columns : {cols}')
print(f'Memory  : {mem_mb:.3f} MB')

In [ ]:
print('Column                     dtype            null_pct  unique')
print('-' * 62)
for col in profile['columns']:
    name     = col['name']
    dtype    = col['dtype']
    null_pct = col['nulls'] / max(rows, 1)
    unique   = col.get('unique', '?')
    print(f'  {name:<24}  {dtype:<15}  {null_pct:.1%}  {unique}')

## Step 2 — Data quality assessment

`assess_csv` runs quality rules over the profiled data and returns:

- **quality_score**: 0.0–1.0 internal scale (displayed as 0–100 below)
- **issues**: typed quality issues with severity, category, and column attribution

In [ ]:
result     = assess_csv(CSV_PATH)
assessment = result['assessment']

score_raw = assessment['quality_score']
score_pct = round(score_raw * 100, 1)
issues    = assessment['issues']

print(f'Quality score : {score_pct}/100')
print(f'Issues found  : {len(issues)}')

severity_counts: dict = {}
for issue in issues:
    sev = issue.get('severity', 'unknown')
    severity_counts[sev] = severity_counts.get(sev, 0) + 1
print()
for sev, cnt in sorted(severity_counts.items()):
    print(f'  {sev:<12}: {cnt}')

In [ ]:
print('Severity    Issue type                              Column')
print('-' * 72)
for issue in issues[:12]:
    sev = issue.get('severity', '?')
    typ = issue.get('type', '?')
    col = issue.get('column') or 'dataset-level'
    print(f'  {sev:<8}  {typ:<38}  {col}')
if len(issues) > 12:
    extra = len(issues) - 12
    print(f'  ... and {extra} more')

## Step 3 — Preprocessing recommendations

`plan_csv` inspects each column and returns actionable recommendations:
null imputation strategy, encoding type, outlier handling, and scaling advice.

No transformations are applied — this is a planning output only.

In [ ]:
plan       = plan_csv(CSV_PATH)
dataset_id = plan['dataset_id']

print(f'Dataset ID: {dataset_id}')
print()
print('Column                     dtype       Issues detected                Recommendations')
print('-' * 100)
for rec in plan['columns']:
    col  = rec['column']
    dt   = rec['dtype']
    iss  = rec['issues']
    recs = rec['recommendations']
    print(f'  {col:<24}  {dt:<10}  {iss:<30}  {recs}')

## Step 4 — Column distribution

`compute_univariate_chart_data` returns binned distribution data for any column —
the same computation that `dqt chart` renders as a terminal bar chart.

Terminal alternative:

```bash
dqt chart demo/sample_orders.csv --column unit_price
```

In [ ]:
import pandas as pd

df    = pd.read_csv(CSV_PATH)
chart = compute_univariate_chart_data(df, 'unit_price')

col_name = chart['column']
col_type = chart['type']
data     = chart['data']

print(f'Column : {col_name}  |  type : {col_type}')
print()

max_count = max((c for _, c in data), default=0)
max_count = max(max_count, 1)
bar_width = 35

print('Bin range                      Count   Distribution')
print('-' * 72)
for label, count in data:
    bar = '#' * int(count / max_count * bar_width)
    print(f'  {label:<28}  {count:>6}  {bar}')

## Interpretation

After running the four steps above:

- **Profile**: structural statistics for every column — dtypes, null counts, and cardinality.
- **Assessment**: a 0–100 quality score and a typed issue list with severity and column attribution.
- **Plan**: per-column preprocessing guidance — imputation, encoding, outlier handling, and scaling.
  No data is modified; use the output to guide ETL or ML feature engineering.
- **Distribution**: a programmatic column histogram matching what `dqt chart` renders in the terminal.

### Quality score guide

| Score | Meaning |
|---|---|
| 90–100 | High quality — minimal issues |
| 75–89  | Acceptable — review flagged columns |
| 50–74  | Needs attention — remediation recommended |
| < 50   | Poor quality — likely pipeline or ingestion problems |

## Next steps

| Goal | How |
|---|---|
| Export star-schema quality artifacts | `dqt export-star demo/sample_orders.csv --outdir dist/` |
| Compare two quality runs | `dqt compare demo/sample_orders.csv --outdir dist/` |
| Create a lineage manifest | `dqt manifest create --run-id <id> --sessions-root <path>` |
| Launch the Streamlit dashboard | `dqt dashboard` |
| Add column-level quality gates | Add `columns.<col>.fail_under` to `dqt.yaml` |
| Python API reference | See `docs/api.md` |
| Full CLI reference | See `docs/cli.md` |